# Session 6 · Automated sleep staging

Scoring a night by hand takes an expert one to two hours. Today a classifier does it in one second: **YASA**'s sleep staging (Vallat & Walker, 2021, *eLife*). The question of the afternoon is not *how* to run it (that's three lines), but **how far you can trust it**, and for what.

You will
1. score **one night** with YASA and compare it with the expert, epoch by epoch
2. measure the agreement: **accuracy, Cohen's κ, confusion matrix, F1 per stage**
3. **experiment with the inputs**: which channel, with or without EOG and EMG, the subject's age
4. use YASA's **confidence** to find the epochs a human should check, and describe each night by the **entropy, intrusions and instability** of its hypnodensity
5. see what **artefacts** and the **rest of the night** do to the scoring
6. compute the **sleep statistics** from the automated hypnogram, for a young and an old subject
7. add the staging to the **cohort pipeline** and rerun all nights
8. answer the cohort question: **with automated scoring, do we get the same age results?**

**How to work**
- Mostly **run, change, look**: cells marked 👉 have a parameter to change and rerun. The `# TODO` cells are short.
- Night: **SC4002** again, then SC4752 (96 y) and the whole cohort.

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py and cohort.py in the repo folder
from sleep_utils import SUBJECTS_CSV, epoch_onsets, load_epochs, load_subject
from cohort import sleep_architecture_features

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=UserWarning)  # LightGBM/sklearn version notes when YASA loads its model

RESULTS_DIR = Path("..") / "results"
subjects = pd.read_csv(SUBJECTS_CSV)

STAGES = ["WAKE", "N1", "N2", "N3", "REM"]
STAGE_COLORS = {"WAKE": "tab:orange", "N1": "tab:cyan", "N2": "tab:blue", "N3": "darkblue", "REM": "tab:red"}

## 1. One night, scored by YASA

**How YASA scores** (details in T8):
- For every 30-s epoch it computes ~30 **features** per channel: standard deviation, relative band power, power ratios, entropy, fractal dimension, … (EEG, and if given EOG and EMG).
- It adds **smoothed** versions of each feature (a 7.5-min centred and a 2-min past rolling average), so every epoch also "sees" its neighbours, like a human scorer looking at the context.
- The smoothed features are **z-scored over the night** (robust z-score: median and IQR).
- A **LightGBM** classifier (gradient-boosted trees) turns them into a probability for each stage. It was trained on ~3,000 nights of the National Sleep Research Resource (NSRR), scored with **AASM** rules, mostly on a **central** EEG channel (C4-M1).

Our data are different: frontal and parietal **bipolar** channels (Fpz-Cz, Pz-Oz), scored with **R&K** rules in 1989–1994. So this is an honest test of a model used outside the data it was trained on, which is what you'll do with your own recordings.

We load the night twice: `load_epochs` for the expert stages and the bad epochs (as on Monday), and `load_subject` for the raw signal. YASA gets the **unfiltered** signal: it filters itself, and its features were computed on unfiltered recordings.

In [ ]:
SUBJECT, NIGHT = 0, 2  # SC4002, 33 years

epochs, stages, artefacts = load_epochs(SUBJECT, NIGHT)  # expert stages + bad epochs (filtered EEG)
raw = load_subject(SUBJECT, NIGHT)  # the same night, lights-off to the end of the sleep period, NOT filtered
onsets = epoch_onsets(epochs)
hours = (onsets - onsets[0]).total_seconds() / 3600  # time since lights-off
print(f"{len(stages)} epochs, channels: {raw.ch_names}")

In [ ]:
hypno = yasa.SleepStaging(raw, eeg_name="Fpz-Cz", eog_name="horizontal").predict()  # a yasa.Hypnogram

auto = hypno.hypno.to_numpy().astype(str)  # YASA's stage of each epoch, as an array of labels like `stages`
proba = hypno.proba  # the probability of each stage in each epoch (each row sums to 1)
assert len(auto) == len(stages)  # one YASA stage per expert epoch, starting at the same time
print(pd.Series(auto).value_counts())
proba.head()

The expert, YASA, and YASA's **hypnodensity**: the probability of each stage across the night (the colours stacked to 1).

In [ ]:
STAGE_Y = {"WAKE": 4, "REM": 3, "N1": 2, "N2": 1, "N3": 0}  # ART and UNS are not drawn


def plot_night(hours, hypnograms, proba=None, title=""):
    """Hypnograms (dict label -> stage array) one above the other, and the hypnodensity below."""
    n = len(hypnograms) + (proba is not None)
    fig, axes = plt.subplots(n, 1, figsize=(14, 1.6 * n + 0.6), sharex=True)
    for ax, (label, h) in zip(axes, hypnograms.items()):
        y = np.array([STAGE_Y.get(s, np.nan) for s in h])
        ax.step(hours, y, where="post", color="black", lw=0.8)
        ax.scatter(hours[y == 3], y[y == 3], s=2, color="tab:red")  # REM in red
        ax.set(yticks=list(STAGE_Y.values()), yticklabels=list(STAGE_Y), ylabel=label)
    if proba is not None:
        axes[-1].stackplot(hours, proba[STAGES].to_numpy().T, colors=[STAGE_COLORS[s] for s in STAGES],
                           labels=STAGES, step="post")
        axes[-1].set(ylim=(0, 1), ylabel="probability")
        axes[-1].legend(ncol=5, loc="lower right", fontsize=8)
    axes[0].set_title(title)
    axes[-1].set_xlabel("Hours since lights-off")
    plt.tight_layout()
    plt.show()


plot_night(hours, {"Expert": stages, "YASA": auto}, proba, title="SC4002")

**Questions**
1. At first sight, does YASA get the night right: the cycles, the N3 at the start, the REM periods?
2. Where in the night, and between which stages, do the two hypnograms disagree most?
3. Where is YASA **unsure** (no colour close to 1 in the hypnodensity)? Is that where it disagrees with the expert?

*Answers*
1. Yes, the architecture is the same: the same four to five cycles, N3 in the first cycles, REM at the end of each cycle.
2. Mostly in the **REM periods**: YASA calls a part of REM **WAKE** or N1 (on SC4002, 42 of 215 REM epochs are called WAKE). Also around the N2/N3 border, and in short awakenings.
3. Mostly yes: the hypnodensity is mixed in the REM periods, at the N2/N3 transitions and around awakenings. Section 4 measures this.

## 2. How good is it?

We compare the two only on the epochs the expert gave a stage (WAKE, N1–N3, REM): YASA never predicts `ART` (movement time) or `UNS` (unscored).

- **Accuracy**: the fraction of epochs where both agree.
- **Cohen's κ**: the agreement corrected for chance, κ = (p_observed − p_chance) / (1 − p_chance). A night with 50 % N2 gives 50 % accuracy to a "classifier" that always says N2, but κ = 0. κ is the number used to compare scorers.

For reference, two **human** experts scoring the same night with AASM rules agree on ~83 % of epochs (Rosenberg & Van Hout, 2013), κ ≈ 0.76 (Danker-Hopfe et al., 2009). Least on N1 (~63 %) and N3 (~67 %).

### Exercise 2.1: accuracy and κ

In [ ]:
scored = np.isin(stages, STAGES)  # leave out ART and UNS
expert, pred = stages[scored], auto[scored]

# TODO: the accuracy, by hand: the fraction of epochs where expert and pred are the same
accuracy = (expert == pred).mean()
# TODO: Cohen's kappa, with cohen_kappa_score from sklearn
kappa = cohen_kappa_score(expert, pred)
print(f"{scored.sum()} epochs: accuracy {accuracy:.2f} (sklearn: {accuracy_score(expert, pred):.2f}), kappa {kappa:.2f}")

The **confusion matrix**: rows = the expert's stage, columns = YASA's. Left in epochs, right in % of each row (= of the expert's epochs of that stage). The diagonal is agreement.

In [ ]:
def plot_confusion(expert, pred, title=""):
    cm = pd.DataFrame(confusion_matrix(expert, pred, labels=STAGES), index=STAGES, columns=STAGES)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, table, fmt in [(axes[0], cm, "d"), (axes[1], 100 * cm.div(cm.sum(axis=1), axis=0), ".0f")]:
        ax.imshow(table, cmap="Blues")
        for i in range(len(STAGES)):
            for j in range(len(STAGES)):
                v = table.iloc[i, j]
                ax.text(j, i, "–" if np.isnan(v) else format(v, fmt), ha="center", va="center",
                        color="white" if v > table.to_numpy()[np.isfinite(table.to_numpy())].max() / 2 else "black")
        ax.set(xticks=range(5), xticklabels=STAGES, yticks=range(5), yticklabels=STAGES, xlabel="YASA",
               ylabel="Expert")
    axes[0].set_title(f"{title}: epochs")
    axes[1].set_title(f"{title}: % of the expert's epochs")
    plt.tight_layout()
    plt.show()


plot_confusion(expert, pred, "SC4002")
print(classification_report(expert, pred, labels=STAGES, digits=2, zero_division=0))

In the report, per stage:
- **recall**: of the expert's epochs of this stage, the fraction YASA also calls this stage (the diagonal of the right matrix)
- **precision**: of YASA's epochs of this stage, the fraction the expert agrees with
- **F1**: the combination of the two (their harmonic mean), the usual one-number summary per stage.

We'll need these numbers many times, so here they are in one function:

In [ ]:
def agreement(expert, pred):
    """Accuracy, Cohen's kappa and the F1 of each stage, as a dict (only the epochs with an expert stage)."""
    expert, pred = np.asarray(expert), np.asarray(pred)
    scored = np.isin(expert, STAGES)
    e, p = expert[scored], pred[scored]
    result = {"accuracy": accuracy_score(e, p), "kappa": cohen_kappa_score(e, p)}
    f1 = f1_score(e, p, labels=STAGES, average=None, zero_division=np.nan)  # NaN if neither has the stage
    result.update({f"F1_{stage}": value for stage, value in zip(STAGES, f1)})
    return result


pd.Series(agreement(stages, auto)).round(2)

**Questions**
1. Accuracy or κ: which is lower, and why?
2. WAKE has a high recall but a low precision. What does that mean? Where do YASA's extra WAKE epochs come from (look at the WAKE column)?
3. Which stage is worst? Compare with the agreement between two humans on that stage.
4. On this night, is YASA as good as a second human scorer?

*Answers*
1. κ (0.74 vs. 0.81 accuracy on SC4002): κ removes the agreement you'd get by chance, given how often each stage occurs.
2. YASA finds almost all of the expert's WAKE (recall 0.88), but only about half of what it calls WAKE is WAKE (precision 0.54): it calls 42 **REM** epochs WAKE. In a frontal EEG with an EOG, REM looks like quiet wake: low-amplitude mixed-frequency EEG, eye movements. What really separates them is the muscle tone (EMG), which we didn't give it (Section 3).
3. **N1** (F1 ≈ 0.46). Humans also agree least on N1 (~63 %). N1 is a transition stage with no clear graphoelement, and in R&K it's a rare stage, so it's also hard to learn.
4. Close: κ = 0.74 vs. ≈ 0.76 between humans. But that's one young, healthy night; Section 6 and 8 show it's not like that everywhere.

## 3. Experiments with the inputs

YASA can use one EEG channel, plus an EOG and an EMG channel, plus the subject's **age and sex** (`metadata`, used as features too). What does each one buy?

👉 Run the table, then add your own rows to `CONFIGS`: e.g. Pz-Oz + EOG, a wrong age (e.g. 90), the other sex, EEG + EMG without EOG.

In [ ]:
def stage(raw, **kwargs):
    """YASA's stage of each epoch (array). kwargs: eeg_name, eog_name, emg_name, metadata."""
    return yasa.SleepStaging(raw, **kwargs).predict().hypno.to_numpy().astype(str)


row = subjects[(subjects["subject"] == SUBJECT) & (subjects["night"] == NIGHT)].iloc[0]
AGE, MALE = int(row["age"]), bool(row["sex_F"] == 2)  # sex_F: 1 = female, 2 = male

CONFIGS = {  # 👉 add your own
    "Fpz-Cz": dict(eeg_name="Fpz-Cz"),
    "Pz-Oz": dict(eeg_name="Pz-Oz"),
    "Fpz-Cz + EOG": dict(eeg_name="Fpz-Cz", eog_name="horizontal"),
    "Fpz-Cz + EOG + EMG": dict(eeg_name="Fpz-Cz", eog_name="horizontal", emg_name="submental"),
    "Fpz-Cz + EOG + age/sex": dict(eeg_name="Fpz-Cz", eog_name="horizontal", metadata=dict(age=AGE, male=MALE)),
}

results = pd.DataFrame({name: agreement(stages, stage(raw, **kwargs)) for name, kwargs in CONFIGS.items()}).T
results.round(2)

**Questions**
1. Which EEG channel works better? Does the EOG help? Which stage gains most from it?
2. The **EMG** should help most for REM (muscle atonia). Does it? Look at the signal below before you answer.
3. Give YASA a wrong age, or the wrong sex. How much does it change? Would you report it in a paper if you didn't know the age?

The submental EMG and Fpz-Cz, 20 s of an N2 epoch:

In [ ]:
T0 = 3600  # 👉 seconds after lights-off
seg = raw.copy().crop(T0, T0 + 20)
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for ax, ch in zip(axes, ["Fpz-Cz", "submental"]):
    ax.plot(seg.times, seg.get_data(picks=[ch], units="uV")[0], lw=0.7, color="black")
    ax.set(ylabel=f"{ch} (µV)")
axes[-1].set_xlabel("s")
plt.show()

*Answers* (SC4002; your own rows will differ)
1. Fpz-Cz (κ 0.71) does better than Pz-Oz (0.61) on its own: Pz-Oz misses most of REM (F1 0.37). With the EOG, κ goes to 0.74; N1 gains most (F1 0.32 → 0.46), then REM (0.67 → 0.72): slow and rapid eye movements. On the cohort (all 153 nights, median κ): Fpz-Cz alone 0.60, + EOG 0.65.
2. Not overall: κ stays at 0.74. REM gets better (F1 0.72 → 0.80) but WAKE and N3 get worse, and on the cohort it's slightly worse (median κ 0.63 vs. 0.65). The plot shows why: the submental channel is a **staircase**, one value per second. In Sleep-EDF the EMG was high-pass filtered, rectified, low-pass filtered and stored as an **envelope sampled at 1 Hz** (the EDF header says 30 samples per 30-s record; MNE resampled it to 100 Hz to put it next to the EEG). It still carries the muscle tone (hence the REM gain), but YASA's EMG features (e.g. power above 30 Hz, entropy) expect a raw EMG at ≥ 100 Hz. A channel can have the right **name** and the wrong **content**: read the dataset's documentation.
3. On SC4002 the right age and sex raise κ from 0.74 to 0.77; on the cohort it makes no difference (median κ 0.65 with or without, 0.64 with a wrong age). If you use metadata, report it, and never feed a made-up age: the staging then depends on a number you invented. On Wednesday we predict age from sleep, so age must **not** go into the staging we build features from (it would leak the answer into the features). That's why the cohort pipeline (Section 7) stages without metadata.

## 4. How sure is YASA?

The **confidence** of an epoch is the probability of the stage YASA chose (the highest probability of the row). If confidence goes with correctness, a human only needs to check the unsure epochs: **semi-automated** scoring.

In [ ]:
confidence = proba.max(axis=1).to_numpy()
correct = stages == auto
bins = pd.cut(confidence, [0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
per_bin = pd.DataFrame({"correct": correct, "bin": bins})[scored].groupby("bin", observed=True)["correct"]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
per_bin.mean().plot.bar(ax=axes[0], color="gray", rot=0, ylim=(0, 1), xlabel="confidence",
                        ylabel="fraction correct", title="Agreement with the expert vs. confidence")
pd.Series(confidence[scored]).groupby(stages[scored]).median().reindex(STAGES).plot.bar(
    ax=axes[1], color=[STAGE_COLORS[s] for s in STAGES], rot=0, ylim=(0, 1), ylabel="median confidence",
    title="Confidence per (expert) stage")
plt.tight_layout()
plt.show()

### Exercise 4.1: keep only the sure epochs

👉 Change `THRESHOLD` and rerun.

In [ ]:
THRESHOLD = 0.8  # 👉

sure = scored & (confidence >= THRESHOLD)
# TODO: kappa on the sure epochs only, and the % of the scored epochs a human would still have to check
kappa_sure = cohen_kappa_score(stages[sure], auto[sure])
pct_to_check = 100 * (1 - sure.sum() / scored.sum())
print(f"confidence >= {THRESHOLD}: kappa {kappa_sure:.2f} on {sure.sum()} epochs, {pct_to_check:.0f} % left to check")

**Questions**
1. Is YASA more often right when it's sure? Which stages is it least sure about?
2. Which threshold gives a κ above the human level? How much of the night would a human then still score?
3. Is a confident epoch always correct? Find a threshold where κ is high, then look at the remaining errors: `pd.crosstab(stages[sure], auto[sure])`.

*Answers* (SC4002)
1. Yes: below 0.4 confidence it agrees with the expert in 32 % of the epochs, between 0.6 and 0.7 in 72 %, above 0.9 in 99 %. Least sure: REM and N1 (median confidence ≈ 0.6), most sure: N3 (0.94) and WAKE (0.88).
2. Threshold 0.7 → κ ≈ 0.89, with 39 % left to check; 0.8 → κ ≈ 0.93, 51 % left. So in this night a human would still look at about half of the epochs, but the easy half is done.
3. No: at ≥ 0.8, 23 of the 501 sure epochs are still wrong (e.g. N3 and REM called WAKE). Confidence is the model's opinion of itself: on data unlike its training data, it can be sure and wrong (Section 8, SC4661: high confidence, κ 0.35).

### Exercise 4.2: entropy, intrusions and instability

The hypnodensity is more than a "how sure" number: it describes **how mixed** each epoch is, and **how it changes** from one epoch to the next. Herzog et al. (2025, *Commun Biol* 8:423, Andrillon lab) used three measures, each computed **within the epochs of one expert-scored stage** (mean and SD per night):

- **Entropy** of an epoch: H = −Σ p_s · log₂ p_s, summed over the 5 stages (in bits). H = 0: one stage has probability 1. H = log₂ 5 = 2.32 bits: all five stages equally likely.
- **Intrusions**: the probability of the *other* state. **Wake intrusion** = mean P(WAKE) in the expert's sleep epochs (N1–REM); **sleep intrusion** = mean P(sleep) = 1 − P(WAKE) in the expert's wake epochs.
- **Instability**: the **KL divergence** between the hypnodensities of two consecutive epochs of the same expert stage: D_KL(P_i ‖ P_i+1) = Σ p_s · log₂(p_s / q_s) (in bits). 0 = the next epoch has exactly the same profile.

In their 927 clinical recordings, insomnia patients who underestimate their sleep had more entropy and more sleep intrusion **in their wake epochs**: the expert scored wake, but the EEG looked partly like sleep.

⚠️ Their hypnodensity came from a classifier trained on each patient's own expert scoring, with calibrated probabilities (Platt scaling). YASA is a pretrained model, not calibrated to our scorers. So compare people **with the same model**, not our numbers with theirs. N1 is left out, as in the paper (few epochs, low agreement).

In [ ]:
HD_STAGES = ["WAKE", "N2", "N3", "REM"]  # the expert stages we describe (N1 left out, as in Herzog et al.)
HD_ORDER = ["WAKE", "N1", "N2", "N3", "REM"]  # the 5 columns of the hypnodensity


def entropy_bits(p):
    """Shannon entropy (bits) of each row of p (n_epochs x 5 stage probabilities)."""
    p = np.clip(p, 1e-12, 1)  # log2(0) is undefined; 0 * log2(0) counts as 0
    # TODO: H = -sum over the 5 stages of p * log2(p), for each epoch (row)
    return -(p * np.log2(p)).sum(axis=1)


def kl_bits(p, q):
    """KL divergence D_KL(p || q) in bits, row by row."""
    p, q = np.clip(p, 1e-12, 1), np.clip(q, 1e-12, 1)
    return (p * np.log2(p / q)).sum(axis=1)


def hypnodensity_features(proba, stages):
    """Entropy, instability and intrusions per expert stage, from YASA's hypnodensity (Herzog et al., 2025).

    proba: DataFrame (n_epochs x WAKE..REM), stages: the expert stage of each epoch.
    H_mean_<stage>, H_sd_<stage>: entropy (bits) of the epochs of this expert stage
    KL_mean_<stage>, KL_sd_<stage>: KL divergence (bits) between consecutive epochs, both of this stage
    wake_intrusion: mean P(WAKE) in the expert's sleep epochs; sleep_intrusion: mean 1 - P(WAKE) in its wake epochs
    """
    p = proba[HD_ORDER].to_numpy()
    stages = np.asarray(stages)
    entropy = entropy_bits(p)
    kl = kl_bits(p[:-1], p[1:])  # epoch i vs. epoch i+1
    features = {}
    for st in HD_STAGES:
        in_stage = stages == st
        pairs = (stages[:-1] == st) & (stages[1:] == st)  # both epochs of the pair in this stage
        features[f"H_mean_{st}"] = entropy[in_stage].mean() if in_stage.any() else np.nan
        features[f"H_sd_{st}"] = entropy[in_stage].std() if in_stage.any() else np.nan
        features[f"KL_mean_{st}"] = kl[pairs].mean() if pairs.any() else np.nan
        features[f"KL_sd_{st}"] = kl[pairs].std() if pairs.any() else np.nan
    sleep, wake = np.isin(stages, ["N1", "N2", "N3", "REM"]), stages == "WAKE"
    features["wake_intrusion"] = p[sleep, 0].mean()  # column 0 = WAKE
    features["sleep_intrusion"] = 1 - p[wake, 0].mean() if wake.any() else np.nan
    return features

In [ ]:
pd.Series(hypnodensity_features(proba, stages)).round(3)  # SC4002

Across the night: the entropy of every epoch and the KL divergence to the next one, with the expert hypnogram.

In [ ]:
p = proba[HD_ORDER].to_numpy()
entropy = entropy_bits(p)
kl = np.r_[kl_bits(p[:-1], p[1:]), np.nan]  # the last epoch has no next one
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True, gridspec_kw={"height_ratios": [2, 2, 1]})
axes[0].plot(hours, entropy, lw=0.7, color="black")
axes[0].set(ylabel="entropy (bits)", ylim=(0, np.log2(5)), title="SC4002")
axes[1].plot(hours, kl, lw=0.7, color="black")
axes[1].set(ylabel="KL to next (bits)", yscale="log")
y = np.array([STAGE_Y.get(s, np.nan) for s in stages])
axes[2].step(hours, y, where="post", color="black", lw=0.8)
axes[2].set(yticks=list(STAGE_Y.values()), yticklabels=list(STAGE_Y), xlabel="Hours since lights-off")
plt.tight_layout()
plt.show()

👉 The same measures for a young and an old night (add others):

In [ ]:
NIGHTS_HD = {"SC4002 (33 y)": (0, 2), "SC4752 (96 y)": (75, 2)}  # 👉

hd = {}
for label, (s, n) in NIGHTS_HD.items():
    _, st, _ = load_epochs(s, n)
    pr = yasa.SleepStaging(load_subject(s, n), eeg_name="Fpz-Cz", eog_name="horizontal").predict().proba
    hd[label] = hypnodensity_features(pr, st)
pd.DataFrame(hd).round(3)

**Questions**
1. In which expert stage is the hypnodensity most mixed (highest entropy)? Least? Does that match the confidence per stage in Exercise 4.1?
2. Where are the peaks of the KL divergence: inside stable bouts, or around stage changes and awakenings?
3. Compare the young and the old night: entropy, instability and the two intrusions. Which differ most?
4. Entropy and intrusions measure how the **model** sees the EEG. If the old night has a higher entropy, is that the brain (less stable sleep, more mixed states), or the model (old EEG is unlike its training data)? How could you tell the two apart?

*Answers* (SC4002 and SC4752)
1. On SC4002: REM is the most mixed (1.37 bits), then N2 (0.90) and WAKE (0.71); N3 the least (0.51). Same order as the confidence in 4.1: entropy and 1 − confidence are almost the same thing per epoch (Spearman ρ = 0.97). Entropy adds that it counts *all* the other stages, not only the winner.
2. Around stage changes and awakenings: the median KL is 0.21 bits for pairs that cross a stage change vs. 0.05 bits for pairs inside the same expert stage. That's why the paper only uses pairs within one stage: otherwise "instability" would mostly count the stage shifts you already have in the hypnogram. Inside a stage, the peaks mark epochs where the profile jumps, e.g. a brief arousal the scorer didn't mark.
3. In the old night, N3 is twice as mixed (1.05 vs. 0.51 bits: the few N3 epochs look half like N2), WAKE is a little more mixed, and wake intrusion into sleep is higher (P(WAKE) during sleep 0.145 vs. 0.094). Sleep intrusion into wake is about the same (0.25 vs. 0.23). REM is *less* mixed in this old night, and instability is not consistently higher: one pair of nights is not a result (Section 8.4).
4. You can't tell from one night. The model was trained on mostly younger, healthier people with other channels, so on old EEG it is less sure **and** less accurate (Section 8). Ways to separate them: check whether the measure still goes with age after accounting for the model's accuracy (κ) (Exercise 8.4); use a model trained or calibrated on old people too; or compare with the human view: the hypnodensity from several human scorers (Anderer et al., 2023, found that human scorers also disagree more on old people's sleep).

## 5. What the input does to the scoring

### 5.1 Artefacts

YASA scored every epoch, including the ones our artefact detector (based on Mariani et al., 2018) + movement time flagged as bad. Is it worse there?

In [ ]:
bad = artefacts["bad"].to_numpy()
# TODO: the accuracy on the clean and on the bad scored epochs (`correct` from Section 4 is True where both agree)
acc_clean = correct[scored & ~bad].mean()
acc_bad = correct[scored & bad].mean()
print(f"clean: {acc_clean:.2f} on {(scored & ~bad).sum()} epochs, bad: {acc_bad:.2f} on {(scored & bad).sum()} epochs")

### 5.2 Does YASA need the whole night?

YASA smooths its features over neighbouring epochs and z-scores them **over the night**. So the stage of an epoch could depend on the rest of the recording. Experiment: give YASA only the first `HOURS` hours, and compare the stages of those epochs with the ones from the whole night.

👉 Try `HOURS = 1, 3, 5`, and `FROM_END = True` (the last hours instead of the first).

In [ ]:
HOURS, FROM_END = 3, False  # 👉

n = int(HOURS * 3600 / 30)  # epochs
if FROM_END:
    part = slice(len(auto) - n, len(auto))
    raw_part = raw.copy().crop(tmin=raw.times[-1] - n * 30 + 1 / raw.info["sfreq"])
else:
    part = slice(0, n)
    raw_part = raw.copy().crop(tmax=n * 30 - 1 / raw.info["sfreq"])
auto_part = stage(raw_part, eeg_name="Fpz-Cz", eog_name="horizontal")
assert len(auto_part) == n

changed = auto_part != auto[part]
print(f"{changed.mean():.0%} of these {n} epochs get another stage than with the whole night")
print("kappa vs. the expert: whole night", round(cohen_kappa_score(stages[part][scored[part]], auto[part][scored[part]]), 2),
      "| only this part", round(cohen_kappa_score(stages[part][scored[part]], auto_part[scored[part]]), 2))
plot_night(hours[part], {"Expert": stages[part], "YASA, whole night": auto[part], f"YASA, {HOURS} h only": auto_part})

**Questions**
1. Is YASA worse on the bad epochs? Should we drop the bad epochs before computing agreement? Before computing sleep statistics?
2. How many epochs change when YASA sees only part of the night? Only near the cut, or everywhere?
3. On Saturday we chose where the night ends (the first wake bout ≥ 3 h). What does 5.2 tell you about that choice for automated staging?

*Answers*
1. On SC4002 yes: 0.82 on clean vs. 0.54 on the 52 bad epochs. (On SC4752 there is no difference: 0.74 vs. 0.76.) But the expert scored the bad epochs too: our rule (Saturday) is that sleep statistics use the **whole** hypnogram, so we compare whole hypnograms. Dropping bad epochs only makes sense for the EEG features.
2. With the first 3 h, about 19 % of the epochs change on SC4002 (10 % on SC4752), not only near the cut: the robust z-score uses the median and IQR of the **whole** recording, so a shorter recording shifts every feature. Agreement with the expert can go up or down.
3. The staging depends on what's in the recording: if the "night" includes the next afternoon, or a long wake period, every epoch's features are scaled differently. The end-of-night rule is part of the method, for YASA too: always stage the same window you analyse, and report it.

## 6. Sleep statistics from the automated hypnogram

In the end we don't want a hypnogram, but numbers: TST, SE, WASO, % of each stage. Agreement per epoch is one thing; are the **statistics** right? `sleep_architecture_features` (from `cohort.py`, Session 4) works on any array of stages.

### Exercise 6.1

In [ ]:
SHOW = ["TIB", "TST", "SE", "SOL", "WASO", "Lat_REM_onset", "%N1", "%N2", "%N3", "%REM", "SCI", "shifts_per_h"]

stats = pd.DataFrame({"expert": sleep_architecture_features(stages), "YASA": sleep_architecture_features(auto)})
# TODO: a column "difference" = YASA - expert
stats["difference"] = stats["YASA"] - stats["expert"]
stats.loc[SHOW].round(1)

### Exercise 6.2: a young and an old night

Everything so far, for any night, in one function. 👉 Add other nights to `NIGHTS` (e.g. the worst ones you'll find in Section 8).

In [ ]:
def compare_night(subject, night):
    """Stage one night with YASA (Fpz-Cz + EOG); agreement and sleep statistics vs. the expert."""
    _, st, _ = load_epochs(subject, night)
    au = stage(load_subject(subject, night), eeg_name="Fpz-Cz", eog_name="horizontal")
    stats_expert, stats_yasa = sleep_architecture_features(st), sleep_architecture_features(au)
    table = pd.Series(agreement(st, au))
    for key in SHOW:
        table[f"{key} expert"], table[f"{key} YASA"] = stats_expert[key], stats_yasa[key]
    return table, st, au


NIGHTS = {"SC4002 (33 y)": (0, 2), "SC4752 (96 y)": (75, 2)}  # 👉
tables = {}
for label, (s, n) in NIGHTS.items():
    tables[label], st, au = compare_night(s, n)
    plot_night(np.arange(len(st)) / 120, {"Expert": st, "YASA": au}, title=label)
pd.DataFrame(tables).round(2)

**Questions**
1. Which statistics does YASA get right, which not? Why is the **REM latency** so fragile?
2. Why are the fragmentation measures (shifts per hour, SCI) off?
3. Compare the young and the old night: κ, and the errors on %N1 and %N3. If YASA makes **different** errors at different ages, what happens to an age effect?

*Answers*
1. TIB is the same by construction. TST, SE and %N2/%N3 are within a few minutes or percent on SC4002. The REM latency is the first REM epoch, so **one** wrong epoch moves it: on SC4002 YASA puts a REM epoch in the first cycle, 20.5 min after sleep onset instead of 66 min. Statistics that depend on a single epoch (latencies) are fragile; sums over the night (TST, % per stage) are robust.
2. Every isolated wrong epoch inside a bout creates two extra stage shifts and splits a bout in two. Shifts per hour go up (15 → 23 on SC4002) and SCI down (4.8 → 3.5 min).
3. κ is lower in the old night (0.60 vs. 0.74). There, YASA halves N1 (21 % → 11 %) and puts it in N2 (65 % → 75 %). If the errors depend on age, the age trend of the automated statistic is different from the true one: a **bias that correlates with the variable you study**. Section 8 checks it on all nights.

## 7. Into the cohort pipeline

The function below does Sections 1, 2, 4.2 and 6 for one night, returns the features as a dict (all YASA features start with `yasa_`, so they don't overwrite the expert's), and saves YASA's hypnogram and hypnodensity per night in `results/<night>/staging_yasa.csv`, for your dashboard.

Test it here on SC4002. Then, in **`cohort_pipeline.ipynb`**:
1. copy **two cells** into new cells **just above** `process_night`: your finished cell with `hypnodensity_features` (Exercise 4.2), then this whole cell
2. in `process_night`, under `# --- Tuesday: automated staging (YASA) vs. the experts`, add the line
   ```python
   features.update(staging_features(subject, night, stages))
   ```
3. test it on SC4002 (the test cell), then rerun all nights (Section 3). The staging adds about 1 s per night: a few minutes for all 153 nights on a recent laptop, up to ~15 min on an older one. **Don't wait for it**: start the run, come back here and go on with Section 8, which uses a precomputed table until your own run is done.

`stages` is the expert hypnogram `load_epochs` already gives inside `process_night`. Your cohort table then has every feature of this notebook, including the entropy, instability and intrusion columns (`yasa_H_…`, `yasa_KL_…`, `yasa_wake_intrusion`, `yasa_sleep_intrusion`).

In [ ]:
# ---- Session 6: automated staging (YASA) vs. the expert ----
import yasa
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from cohort import sleep_architecture_features
from sleep_utils import epoch_onsets, load_subject, night_code

STAGING_STAGES = ["WAKE", "N1", "N2", "N3", "REM"]


def staging_features(subject, night, stages, results_dir=Path("..") / "results"):
    """YASA's staging (Fpz-Cz + EOG, no EMG, no metadata) vs. the expert hypnogram `stages`, as a dict.

    yasa_kappa, yasa_accuracy, yasa_F1_<stage>, yasa_confidence (mean probability of the chosen stage), and the
    sleep statistics of the YASA hypnogram (yasa_TST, yasa_%N3, ...). Saves results/<night>/staging_yasa.csv.
    """
    raw = load_subject(subject, night)  # unfiltered: YASA filters itself
    hypno = yasa.SleepStaging(raw, eeg_name="Fpz-Cz", eog_name="horizontal").predict()
    auto = hypno.hypno.to_numpy().astype(str)
    stages = np.asarray(stages)
    assert len(auto) == len(stages), "YASA and the expert hypnogram have different numbers of epochs"

    scored = np.isin(stages, STAGING_STAGES)  # the expert's ART and UNS epochs are left out
    e, p = stages[scored], auto[scored]
    features = {"yasa_accuracy": accuracy_score(e, p), "yasa_kappa": cohen_kappa_score(e, p),
                "yasa_confidence": hypno.proba.max(axis=1).mean()}
    f1 = f1_score(e, p, labels=STAGING_STAGES, average=None, zero_division=np.nan)
    features.update({f"yasa_F1_{s}": v for s, v in zip(STAGING_STAGES, f1)})
    features.update({f"yasa_{k}": v for k, v in sleep_architecture_features(auto).items()})
    # Exercise 4.2: entropy, instability and intrusions per expert stage (needs hypnodensity_features)
    features.update({f"yasa_{k}": v for k, v in hypnodensity_features(hypno.proba, stages).items()})

    table = hypno.proba.add_prefix("P_").reset_index(drop=True)
    table.insert(0, "onset", hypno.hypno.index.strftime("%H:%M:%S"))
    table.insert(1, "expert", stages)
    table.insert(2, "yasa", auto)
    folder = Path(results_dir) / night_code(subject, night)
    folder.mkdir(parents=True, exist_ok=True)
    table.to_csv(folder / "staging_yasa.csv", index=False)
    return features

In [ ]:
test = staging_features(0, 2, stages)  # SC4002: yasa_kappa should be 0.74
pd.Series(test).round(2)

## 8. The cohort: same age results with automated scoring?

We read the cohort table and keep the nights you included in `database.csv`. As long as your own cohort run (Section 7) isn't finished, this uses the **precomputed** table in `precomputed/cohort_staging.csv` (all nights, the same code). When your run is done, rerun this section: it switches to your own table.

In [ ]:
USE_PRECOMPUTED = False  # 👉 True: use the precomputed table even if your own run is there (e.g. to compare)

own_file = RESULTS_DIR / "cohort_features.csv"
own = pd.read_csv(own_file) if own_file.exists() else pd.DataFrame()
if "yasa_kappa" in own and not USE_PRECOMPUTED:
    cohort, source = own, "your own cohort run"
else:
    cohort, source = pd.read_csv(Path("..") / "precomputed" / "cohort_staging.csv"), "the PRECOMPUTED table"

database = pd.read_csv(Path("..") / "database.csv")
included = database.loc[database["include"].astype(str).str.lower().isin(["true", "1"]), "code"]
cohort = cohort[cohort["code"].isin(included)].reset_index(drop=True)
print(f"Using {source}: {len(cohort)} nights, {cohort['subject'].nunique()} people")
cohort["yasa_kappa"].describe().round(2)

### Exercise 8.1: agreement vs. age

In [ ]:
METRIC = "yasa_kappa"  # 👉 also try yasa_F1_N1, yasa_F1_N3, yasa_F1_REM, yasa_confidence

# TODO: Spearman correlation of METRIC with age (nan_policy="omit": some nights have no N3)
rho, p = spearmanr(cohort["age"], cohort[METRIC], nan_policy="omit")

fig, ax = plt.subplots(figsize=(7, 4))
for night, marker in [(1, "o"), (2, "s")]:
    m = cohort["night"] == night
    ax.scatter(cohort.loc[m, "age"], cohort.loc[m, METRIC], marker=marker, s=20, alpha=0.7, label=f"night {night}")
ax.set(xlabel="Age (years)", ylabel=METRIC, title=f"{METRIC} vs. age: ρ = {rho:.2f}, p = {p:.1g}")
ax.legend()
plt.show()
cohort.nsmallest(5, "yasa_kappa")[["code", "age", "yasa_kappa", "yasa_confidence", "pct_bad"]].round(2)

### Exercise 8.2: Bland–Altman

For one statistic: x = the mean of expert and YASA, y = their difference (YASA − expert), one dot per night, coloured by age. The **bias** is the mean difference; the **limits of agreement** are bias ± 1.96 SD (95 % of nights fall inside).

👉 Try `TST`, `SE`, `WASO`, `%N1`, `%N3`, `%REM`, `Lat_REM_onset`.

In [ ]:
STAT = "%N1"  # 👉

expert_val, yasa_val = cohort[STAT], cohort[f"yasa_{STAT}"]
mean, diff = (expert_val + yasa_val) / 2, yasa_val - expert_val
# TODO: the bias (mean of diff) and the lower and upper limits of agreement (bias ± 1.96 standard deviations)
bias = diff.mean()
low, high = bias - 1.96 * diff.std(), bias + 1.96 * diff.std()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sc = axes[0].scatter(mean, diff, c=cohort["age"], cmap="viridis", s=20)
for y, ls in [(bias, "-"), (low, "--"), (high, "--")]:
    axes[0].axhline(y, color="gray", ls=ls)
axes[0].axhline(0, color="black", lw=0.5)
axes[0].set(xlabel=f"mean of expert and YASA: {STAT}", ylabel="YASA − expert",
            title=f"Bland–Altman, {STAT}: bias {bias:.1f}, limits [{low:.1f}, {high:.1f}]")
plt.colorbar(sc, ax=axes[0], label="age")
axes[1].scatter(cohort["age"], diff, s=20, color="gray")
axes[1].axhline(0, color="black", lw=0.5)
rho_d, p_d = spearmanr(cohort["age"], diff, nan_policy="omit")
axes[1].set(xlabel="Age (years)", ylabel="YASA − expert", title=f"Is the error age-dependent? ρ = {rho_d:.2f}")
plt.tight_layout()
plt.show()

### Exercise 8.3: would the study come to the same conclusions?

The age trend (Spearman ρ with age) of each statistic, from the expert hypnograms and from YASA's.

In [ ]:
STATS = ["TST", "SE", "SOL", "WASO", "Lat_REM_onset", "%N1", "%N2", "%N3", "%REM", "SCI", "shifts_per_h"]

rows = {}
for stat in STATS:
    rows[stat] = {}
    for source, col in [("expert", stat), ("YASA", f"yasa_{stat}")]:
        rho, p = spearmanr(cohort["age"], cohort[col], nan_policy="omit")
        rows[stat][f"ρ {source}"], rows[stat][f"p {source}"] = rho, p
trends = pd.DataFrame(rows).T
trends[["ρ expert", "ρ YASA", "p expert", "p YASA"]].round(3)

**Questions**
1. Is YASA worse in older people? Which stage drives it (try the F1 metrics)? Give two reasons why the old nights are harder.
2. Look at the five worst nights: put one into Section 6.2 (`NIGHTS`) and look at the hypnograms. What went wrong?
3. In the Bland–Altman plots: which statistic has the largest bias? Is the error the same at every age?
4. Which age trends survive with YASA? Which one disappears, or gets stronger? So: could this study have been done without human scorers? What would you write in the methods?
5. 🐭 Rodent staging tools (e.g. AccuSleep, SPINDLE) are trained on one lab's mice and scorers. What does this session tell you about using them on your own recordings?

*Answers* (all 153 nights; yours change with your exclusions)
1. Yes, κ falls with age (ρ = −0.39; median 0.70 below 40 y, 0.58 above 80 y), and so does YASA's confidence (ρ = −0.46). The F1 that drops most with age is **REM** (ρ = −0.48; median 0.89 below 40 y, 0.67 above 80 y), then N3 (rare in old nights) and N1. Reasons: more N1 and wake, the hardest stages, and less N3, the easiest; lower EEG amplitude in old age; few very old people in the training data (domain shift); and, see question 2, the expert scoring itself.
2. The worst nights are old people (SC4611 101 y, SC4762 90 y, SC4661 88 y, SC4602 89 y; and SC4511, 70 y). In SC4611 and SC4762 the expert scored **~40 % of the night as REM** (399 of 1010 epochs in SC4611), about twice the usual share and very unusual at that age. YASA calls most of it N2 or WAKE. Who is right? Look at the EOG (rapid eye movements?) and the EMG envelope in `raw.plot()` (optional exercise 1). Disagreement does not tell you which scorer is wrong: the expert is the reference, not the truth.
3. %N1: bias −8 points (YASA scores about half of the expert's N1), and the error **grows with age** (ρ = −0.52 between age and YASA − expert). %N3: +7 points at all ages (R&K stage 3/4 needs ≥ 20 % of the epoch with slow waves > 75 µV; YASA learnt AASM N3 on other montages). TST: −28 min, limits −86 to +30 min: fine for a group mean, not for one person. %N2: almost no overall bias, but an age-dependent one (ρ = 0.55): the old people's N1 ends up in N2.
4. SE, WASO and %N3 keep their trends (ρ ≈ −0.55/−0.58, 0.61/0.57, −0.39/−0.37). But **%N1 vs. age disappears** (ρ 0.57 → 0.09), a **%N2 vs. age trend appears** out of nothing (0.00 → 0.58), %REM decline looks stronger (−0.31 → −0.53), and TST vs. age becomes "significant" (p 0.11 → 0.01). So: fine for robust totals, but two of the conclusions would be wrong, and both are errors that correlate with age. Rule: validate an automated scorer **against human scoring in your own population**, especially when the effect you study (age, disease, a drug) changes the stages the scorer is worst at. In the methods: which scorer, version, channels, the agreement on a human-scored subset (κ, per-stage F1, Bland–Altman), and for which statistics you trust it.
5. The same: the model's training data (species, strain, montage, scorer, scoring rules) set where it can be trusted. Check κ and Bland–Altman on a part of your own data scored by a human before trusting it, and recheck after any change (new mice, new genotype, new electrode).

### Exercise 8.4: entropy, intrusions and instability vs. age

The hypnodensity measures of Exercise 4.2, for every night. 👉 Plot any of them against age, and against YASA's κ.

In [ ]:
HD_COLS = [c for c in cohort if c.startswith(("yasa_H_mean", "yasa_KL_mean")) or c.endswith("_intrusion")]
table = {}
for col in HD_COLS:
    rho_age, p_age = spearmanr(cohort["age"], cohort[col], nan_policy="omit")
    rho_k, _ = spearmanr(cohort["yasa_kappa"], cohort[col], nan_policy="omit")
    table[col] = {"median": cohort[col].median(), "ρ age": rho_age, "p age": p_age, "ρ with kappa": rho_k}
pd.DataFrame(table).T.round(3)

In [ ]:
METRIC = "yasa_H_mean_N2"  # 👉 any column of the table above

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(cohort["age"], cohort[METRIC], s=15, c=cohort["yasa_kappa"], cmap="viridis")
axes[0].set(xlabel="Age (years)", ylabel=METRIC, title="colour = YASA's kappa")
axes[1].scatter(cohort["yasa_kappa"], cohort[METRIC], s=15, c=cohort["age"], cmap="viridis")
axes[1].set(xlabel="YASA's kappa vs. the expert", ylabel=METRIC, title="colour = age")
plt.tight_layout()
plt.show()

**Questions**
1. Which hypnodensity measures change with age? In which direction? Is that what you'd expect from what you know about sleep in old age (T2)?
2. The same measures also correlate with YASA's κ. Why is that a problem for the interpretation in question 1? How would you separate "less stable sleep" from "the model is worse on old EEG"?
3. Would you add these measures to Wednesday's age prediction? What would you write about them in the methods?

*Answers* (all 153 nights)
1. Entropy goes up with age in every stage: N3 (ρ = 0.53), N2 (0.40), REM (0.38), WAKE (0.22); wake intrusion into sleep too (0.37). So in older people the expert-scored stages look more **mixed** to the model, and sleep looks more like wake: it fits lighter, more fragmented sleep in old age (T2) and the finding of Anderer et al. (2023) that sleep becomes more ambiguous with age. Instability in N2 goes **down** with age (ρ = −0.34): old N2 is uniformly mixed from epoch to epoch, while young N2 alternates between clear N2 and N3-like epochs. Sleep intrusion into wake barely changes (ρ = 0.15, p = 0.06).
2. They also go with κ (e.g. N2 entropy ρ = −0.42, wake intrusion −0.50): where YASA is wrong, it is also unsure. So part of the age effect may be the model failing on old EEG, not the brain. One check: the partial correlation with age after removing κ (ranks): N3 entropy ρ = 0.46, N2 entropy 0.28, REM 0.26, wake intrusion 0.22, N2 instability −0.36, all p < 0.01. So the age effects get smaller but don't disappear. Better tests: a model calibrated on old people, or several human scorers.
3. They are tempting for age prediction (they change with age and are not just the stage percentages). But they depend on one specific model: report the model and version, the channels, that the probabilities are not calibrated, and that the measures are computed per expert-scored stage. And never compare their absolute values with another model's (e.g. the paper's 0.74–1.27 bits for wake): only within the same pipeline.

## Optional exercises

1. **See the disagreements in the raw signal.** Add YASA's stages as annotations to `raw` (only where it disagrees with the expert), and browse them with `raw.plot()`. Would you side with the expert or with YASA?
2. **What becomes of the expert's N1?** Using the expert and YASA stage of every epoch of every night (your `results/<night>/staging_yasa.csv` files, or `precomputed/staging_epochs.csv.gz`), compute what YASA calls the expert's N1 epochs (in %), for people below 40 and above 80. Same for the expert's REM.

In [ ]:
# Optional exercise 1
disagree = np.flatnonzero(scored & ~correct)
yasa_annotations = mne.Annotations(onset=raw.first_time + disagree * 30, duration=30,
                                   description=[f"YASA {auto[i]}" for i in disagree],
                                   orig_time=raw.annotations.orig_time)
raw_view = raw.copy().set_annotations(raw.annotations + yasa_annotations)
raw_view.plot(picks=["Fpz-Cz", "Pz-Oz", "horizontal", "submental"], duration=60, time_format="clock",
              scalings=dict(eeg=50e-6, eog=100e-6, emg=1e-5, misc=1))

In [ ]:
# Optional exercise 2
files = {code_: RESULTS_DIR / code_ / "staging_yasa.csv" for code_ in cohort["code"]}
if all(f.exists() for f in files.values()) and not USE_PRECOMPUTED:  # your own run
    epochs_all = pd.concat([pd.read_csv(f, usecols=["expert", "yasa"]).assign(code=c) for c, f in files.items()])
else:
    epochs_all = pd.read_csv(Path("..") / "precomputed" / "staging_epochs.csv.gz")
    epochs_all = epochs_all[epochs_all["code"].isin(cohort["code"])]
epochs_all = epochs_all.merge(cohort[["code", "age"]], on="code")
epochs_all["group"] = pd.cut(epochs_all["age"], [0, 40, 80, 120], labels=["< 40 y", "40-80 y", "> 80 y"])
for expert_stage in ["N1", "REM"]:
    sub = epochs_all[epochs_all["expert"] == expert_stage]
    print(f"Expert {expert_stage}: what YASA calls it (% per age group)")
    print((100 * pd.crosstab(sub["group"], sub["yasa"], normalize="index")).round(0), "\n")

*Answers* (optional exercise 2, all 153 nights)
- Expert N1 below 40 y: YASA calls 32 % N1, 41 % WAKE, 14 % REM, 13 % N2. Above 80 y: 18 % N1, 33 % WAKE, **41 % N2**. In the young, N1 is lost to wake; in the old, to N2. That is the %N2 trend with age in Exercise 8.3.
- Expert REM: YASA agrees on 88 % below 40 y but only 57 % above 80 y (22 % N2, 15 % WAKE), mostly from the few nights with ~40 % expert REM.

## What's next

The staging columns are now in the cohort table. Tomorrow: **spindles and slow waves** (on the clean N2/N3 epochs), then **age prediction**: from the expert's features, and (optional) from YASA's, to see whether automated scoring changes the predicted age.